# Tutorial 14 -- Kerr Free Evolution

Prepare a coherent cavity state, let it evolve under self-Kerr in the matched rotating frame, and inspect the resulting phase-space dynamics.

**Prerequisites.** Tutorials 03, 05, and 08 are recommended first.


## 1. Goal

We will isolate self-Kerr evolution in the storage mode and visualize how the cavity moments and Wigner function change over time.


## 2. Physical Background

In the matched rotating frame, a cavity with self-Kerr no longer undergoes a large bare rotation, but it still accumulates number-dependent phase. That phase bends the coherent-state trajectory away from rigid harmonic motion.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    ns,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
total_time = 20.0 * us
dt = 0.2 * us


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.05),
    omega_q=GHz(6.25),
    alpha=MHz(-220.0),
    chi=0.0,
    kerr=MHz(-0.080),
    n_cav=28,
    n_tr=2,
)
frame = FrameSpec(omega_c_frame=model.omega_c, omega_q_frame=model.omega_q)
initial_state = prepare_state(
    model,
    StatePreparationSpec(qubit=qubit_state("g"), storage=coherent_state(1.8)),
)


## 6. Pulse / Sequence Construction


In [ ]:
compiled = SequenceCompiler(dt=dt).compile([], t_end=total_time)


## 7. Running the Simulation


In [ ]:
result = simulate_sequence(
    model,
    compiled,
    initial_state,
    {},
    config=SimulationConfig(frame=frame, store_states=True, max_step=dt),
)
cavity_means = np.array([mode_moments(state, "storage")["a"] for state in result.states], dtype=np.complex128)
photon_numbers = np.array([mode_moments(state, "storage")["n"] for state in result.states], dtype=float)


## 8. Visualizing the Results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4))
axes[0].plot(np.real(cavity_means), np.imag(cavity_means), "o-")
axes[0].set_xlabel(r"Re$\langle a \rangle$")
axes[0].set_ylabel(r"Im$\langle a \rangle$")
axes[0].set_title("Coherent-state trajectory under self-Kerr")

axes[1].plot(compiled.tlist / us, photon_numbers)
axes[1].set_xlabel("Time [us]")
axes[1].set_ylabel(r"$\langle n \rangle$")
axes[1].set_title("Mean cavity occupation during Kerr evolution")
plt.show()

final_rho_c = reduced_cavity_state(result.final_state)
xvec, yvec, w = cavity_wigner(final_rho_c, n_points=81, extent=4.5)
fig, ax = plt.subplots()
image = ax.imshow(w, origin="lower", extent=[xvec[0], xvec[-1], yvec[0], yvec[-1]], cmap="RdBu_r", aspect="equal")
ax.set_xlabel("x")
ax.set_ylabel("p")
ax.set_title("Final cavity Wigner function after Kerr evolution")
fig.colorbar(image, ax=ax, shrink=0.86)
plt.show()


## 9. Physical Interpretation

The cavity photon number stays nearly constant because there is no loss term here. The interesting physics is the nonlinear phase accumulation, which shears the phase-space distribution and bends the coherent-state trajectory.


## 10. Exercises / Next Steps

- Reverse the sign of the Kerr coefficient and compare the direction of the phase-space bending.
- Add a small cavity decay rate and observe how loss and Kerr compete.
- Continue to Tutorials 15 and 16 for cross-Kerr and lossy bosonic dynamics.
